In [5]:
print("hellO")

hellO


In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = r"E:\Nihal\RAG_Projects\production_rag\production-rag\data\documents\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()




C:\Users\Asus\AppData\Local\Temp\ipykernel_13480\2844796311.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
print(len(docs))
print(docs[0].metadata)

564
{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-05-16T09:54:54+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'total_pages': 564, 'page': 0, 'page_label': 'Cover'}


In [8]:
from langchain_core.documents import Document
def filter_docs(docs):
    filtered_docs = []
    
    for doc in docs:
        src = doc.metadata.get("source")
        filtered_docs.append(Document(page_content=doc.page_content, metadata={"source": src,"page": doc.metadata.get("page"),"title": doc.metadata.get("title")}))
    return filtered_docs

In [9]:
filtered_docs = filter_docs(docs)
print(filtered_docs[0].page_content)

Aurélien Géron
Hands-On  
Machine Learning  
with Scikit-Learn  
& TensorFlow  
CONCEPTS, TOOLS, AND TECHNIQUES  
TO BUILD INTELLIGENT SYSTEMS
 D o w n l o a d   f r o m   f i n e l y b o o k   w w w . f i n e l y b o o k . c o m


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(filtered_docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(filtered_docs)
    return chunks


In [11]:
chunks = split_docs(filtered_docs)
print(len(chunks))
print(chunks[0].metadata)
print(chunks[1])

1527
{'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'page': 0, 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow'}
page_content=' D o w n l o a d   f r o m   f i n e l y b o o k   w w w . f i n e l y b o o k . c o m' metadata={'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'page': 1, 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow'}


In [12]:
from sentence_transformers import SentenceTransformer

def embed_docs(chunks):
    model = SentenceTransformer("all-MiniLM-L6-v2")

    cleaned_texts = []
    metadata = []

    for chunk in chunks:
        text = chunk.page_content

        # Remove null bytes
        text = text.replace("\x00", "")

        # Remove problematic unicode characters
        text = text.encode(
            "utf-8",
            errors="ignore"
        ).decode("utf-8")

        cleaned_texts.append(text)
        metadata.append(chunk.metadata)

    embeddings = model.encode(
        cleaned_texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    return embeddings,cleaned_texts,metadata

In [ ]:
def embed_docs(chunks):
    model = SentenceTransformer("all-MiniLM-L6-v2")

    cleaned_texts = []
    metadata = []

    for chunk in chunks:
        text = chunk.page_content

        # Remove null bytes
        text = text.replace("\x00", "")

        # Remove problematic unicode characters
        text = text.encode(
            "utf-8",
            errors="ignore"
        ).decode("utf-8")

        cleaned_texts.append(text)
        metadata.append(chunk.metadata)

    embeddings = model.encode(
        cleaned_texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    return embeddings

In [13]:
embeddings, texts, metadata = embed_docs(chunks)

Batches: 100%|██████████| 48/48 [00:46<00:00,  1.03it/s]


In [14]:
print(len(embeddings))
print(len(texts))
print(embeddings[0])
print(texts[0])

1527
1527
[-5.21549992e-02 -4.66670394e-02  2.01457795e-02 -2.12761816e-02
  6.22026622e-02 -1.00793652e-01  3.31360148e-03 -3.11944205e-02
 -1.24043062e-01 -1.78471226e-02 -1.83520524e-03 -7.50954356e-03
 -6.92526177e-02 -3.34472768e-02 -1.36123709e-02  2.95399013e-03
 -1.82151627e-02 -2.25401483e-03 -1.08759210e-03 -5.16160987e-02
 -2.40503028e-02  3.78625020e-02 -3.15631297e-03  5.21167368e-02
  4.77836654e-02 -2.81028245e-02  7.57137686e-02  1.33545613e-02
  1.09864101e-02 -2.41896156e-02  4.47967090e-02 -1.61413774e-02
 -1.48746604e-02 -1.40493400e-02 -8.94415751e-02  5.95860509e-03
  3.53959873e-02 -2.82838363e-02  7.17336079e-05  3.54922861e-02
 -5.70779201e-03 -4.86543253e-02  3.48108150e-02  3.03983558e-02
  6.41345382e-02  3.35239843e-02 -6.08638152e-02 -1.23857036e-01
  3.74558195e-02 -1.58005208e-02 -1.14358239e-01 -6.13473989e-02
 -3.20628770e-02  5.16274795e-02 -2.82257963e-02  6.04149178e-02
  2.94422600e-02 -7.85633773e-02 -3.93659361e-02 -2.28605065e-02
  1.32035455e-0

In [15]:
ids = [
    f"chunk_{i}"
    for i in range(len(chunks))
  ]

In [16]:
# Store in Chroma
from app.database.vectordb import ChromaVectorDB


vectordb = ChromaVectorDB()

vectordb.add_documents(
    ids=ids,
    texts=texts,
     embeddings=embeddings,
    metadatas=metadata
)



In [17]:
query = "Explain the concept of overfitting in machine learning and how it can be mitigated."
model = SentenceTransformer("all-MiniLM-L6-v2")
query = "Explain ridge and lasso regression in machine learning and their differences."

query_embedding = model.encode(query, convert_to_numpy=True)

In [18]:
results = vectordb.search(
    query_embedding
)

for doc in results["documents"][0]:

    print("=" * 80)

    print(doc[:500])

    print("\n")


Lasso Regression
Leas
t Absolute Shrinkage and Selection Operator Regression (simply called Lasso
Regression) is another regularized version of Linear Regression: just like Ridge
Regression, it adds a regularization term to the cost function, but it uses the ℓ 1 norm
of the weight vector instead of half the square of the ℓ2 norm (see Equation 4-10).
Equation 4-10. Lasso Regression cost function
J θ = MSE θ + α ∑
i = 1
n
θi
Figure 4-18 shows the same thing as Figure 4-17 but replaces Ridge models


As with Linear Regression, we can perform Ridge Regression either by computing a 
closed-form equation or by performing Gradient Descent. The pros and cons are the
same. Equation 4-9 shows the closed-form solution (where A is the n × n identity
matrix13 except with a 0 in the top-left cell, corresponding to the bias term).
128 | Chapter 4: Training Models
Download from finelybook www.finelybook.com


Stochastic Gradient Descent                                                                  

In [19]:
for metadata in results["metadatas"][0]:
    print(metadata)

{'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'page': 151}
{'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'page': 149, 'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf'}
{'page': 6, 'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow'}
{'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'page': 149, 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow'}
{'page': 149, 'title': 'Hands-On Machine Learning with Scikit-

In [20]:
from dotenv import load_dotenv
load_dotenv()
import os



In [21]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [22]:
index_name = "ml-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
        

)

In [23]:
index = pc.Index(index_name)

In [37]:
from langchain_pinecone import PineconeVectorStore
index = pc.Index(index_name)

docsearch = PineconeVectorStore.from_documents(
    index_name=index_name,
    embedding=model,
    documents=chunks,
   
)



AttributeError: 'SentenceTransformer' object has no attribute 'embed_documents'

In [30]:
print(chunks[0].metadata)

{'source': 'E:\\Nihal\\RAG_Projects\\production_rag\\production-rag\\data\\documents\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'page': 0, 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow'}
